[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module2/07-async-await.ipynb)

# Async / Await
**Module 2 — Intermediate Python | Estimated time: 30 minutes**

## Learning Objectives
- Understand the **asyncio event loop** and how it schedules coroutines
- Write `async def` functions and use the `await` keyword
- Distinguish **coroutines** from regular functions and threads
- Run multiple tasks concurrently with **`asyncio.gather()`**
- Compare `asyncio.sleep()` vs `time.sleep()` for blocking behaviour
- Make async HTTP requests with **`aiohttp`**
- Implement a **producer/consumer** pattern with `asyncio.Queue`
- Choose between **`ThreadPoolExecutor`** and asyncio for different workload types

In [ ]:
!pip install aiohttp --quiet
import asyncio
import time
import sys
print('Python:', sys.version)
print('Setup complete.')

## 1. Coroutines vs Regular Functions

A coroutine is defined with `async def`. Calling it does **not** run the body — it returns a coroutine object.  
The body only runs when the coroutine is *awaited* or scheduled on the event loop.

In [ ]:
import asyncio

# Regular function — runs immediately when called
def regular_greet(name: str) -> str:
    return f'Hello, {name}!'

# Coroutine function — returns a coroutine object when called
async def async_greet(name: str) -> str:
    return f'Hello, {name}! (async)'


# Calling the regular function runs it:
print(regular_greet('World'))          # 'Hello, World!'

# Calling the coroutine does NOT run it:
coro = async_greet('World')
print(type(coro))                      # <class 'coroutine'>
print(repr(coro))                      # coroutine object — body not yet executed

# We must await it (inside async context) or use asyncio.run():
result = asyncio.run(async_greet('World'))
print(result)                          # 'Hello, World! (async)'
coro.close()  # clean up the unawaited coroutine to silence the warning

## 2. `asyncio.sleep` vs `time.sleep` — Blocking vs Non-Blocking

`time.sleep()` **blocks** the entire thread — no other coroutine can run.  
`asyncio.sleep()` **yields control** back to the event loop, letting other coroutines proceed.

In [ ]:
import asyncio
import time

# --- Blocking with time.sleep ---
def blocking_worker(name: str, delay: float):
    print(f'{name}: starting (blocking)')
    time.sleep(delay)         # blocks the whole thread
    print(f'{name}: done')

start = time.perf_counter()
blocking_worker('A', 0.3)
blocking_worker('B', 0.3)
blocking_worker('C', 0.3)
print(f'Sequential (blocking): {time.perf_counter() - start:.2f}s  (expected ~0.9s)\n')


# --- Non-blocking with asyncio.sleep ---
async def async_worker(name: str, delay: float):
    print(f'{name}: starting (async)')
    await asyncio.sleep(delay)   # yields to event loop
    print(f'{name}: done')

async def run_all():
    start = time.perf_counter()
    # gather() runs all three coroutines concurrently
    await asyncio.gather(
        async_worker('A', 0.3),
        async_worker('B', 0.3),
        async_worker('C', 0.3),
    )
    print(f'Concurrent (async):   {time.perf_counter() - start:.2f}s  (expected ~0.3s)')

asyncio.run(run_all())

## 3. `asyncio.gather()` — Concurrent Tasks

`gather()` schedules multiple coroutines and returns when all of them have completed.

In [ ]:
import asyncio
import time
import random

async def fetch_price(symbol: str) -> dict:
    """Simulate fetching a stock price from a remote API."""
    delay = random.uniform(0.1, 0.5)    # variable network latency
    await asyncio.sleep(delay)
    price = random.uniform(100, 500)
    return {'symbol': symbol, 'price': round(price, 2), 'latency_ms': int(delay * 1000)}


async def fetch_portfolio(symbols: list[str]) -> list[dict]:
    """Fetch all prices concurrently."""
    start = time.perf_counter()
    results = await asyncio.gather(*[fetch_price(s) for s in symbols])
    elapsed = time.perf_counter() - start
    print(f'Fetched {len(symbols)} prices in {elapsed:.3f}s (concurrent)')
    return list(results)


symbols = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA', 'NVDA']
results = asyncio.run(fetch_portfolio(symbols))
for r in results:
    print(f"  {r['symbol']:5s}  ${r['price']:>7.2f}  (latency: {r['latency_ms']}ms)")

## 4. `asyncio.create_task()` and Task Management

`create_task()` schedules a coroutine immediately (it starts running even before `await`), giving you more control than `gather()`.

In [ ]:
import asyncio
import time

async def step(name: str, duration: float) -> str:
    print(f'  [{name}] started')
    await asyncio.sleep(duration)
    print(f'  [{name}] finished after {duration}s')
    return f'result_from_{name}'


async def main():
    print('Creating tasks...')
    task_a = asyncio.create_task(step('A', 0.3))   # starts immediately
    task_b = asyncio.create_task(step('B', 0.1))   # starts immediately
    task_c = asyncio.create_task(step('C', 0.2))   # starts immediately

    print('Tasks created — now awaiting them...')
    # Await in any order; they all run concurrently
    result_b = await task_b
    result_a = await task_a
    result_c = await task_c

    print(f'Results: {result_a}, {result_b}, {result_c}')


start = time.perf_counter()
asyncio.run(main())
print(f'Total time: {time.perf_counter() - start:.2f}s  (expected ~0.3s, not 0.6s)')

## 5. Async HTTP with `aiohttp`

`aiohttp` is the standard async HTTP client. It lets you fire many requests in parallel without blocking.

In [ ]:
import asyncio
import aiohttp
import time

URLS = [
    'https://httpbin.org/delay/1',
    'https://httpbin.org/get?q=python',
    'https://httpbin.org/uuid',
    'https://httpbin.org/ip',
]

async def fetch(session: aiohttp.ClientSession, url: str) -> dict:
    """Fetch a URL and return status + first 80 chars of body."""
    async with session.get(url, timeout=aiohttp.ClientTimeout(total=15)) as resp:
        text = await resp.text()
        return {'url': url, 'status': resp.status, 'preview': text[:80].replace('\n', ' ')}


async def fetch_all(urls: list[str]) -> list[dict]:
    # aiohttp recommends re-using a single ClientSession for the whole run
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, url) for url in urls]
        return await asyncio.gather(*tasks, return_exceptions=True)


print('Fetching URLs concurrently with aiohttp...')
start = time.perf_counter()
results = asyncio.run(fetch_all(URLS))
elapsed = time.perf_counter() - start

for r in results:
    if isinstance(r, Exception):
        print(f'  ERROR: {r}')
    else:
        print(f"  [{r['status']}] {r['url'].split('/')[-1]:20s}  {r['preview'][:60]}")

print(f'\nTotal: {elapsed:.2f}s for {len(URLS)} requests in parallel')

## 6. `asyncio.Queue` — Producer / Consumer Pattern

Queues let producers and consumers run at different speeds without tight coupling.

In [ ]:
import asyncio
import random
import time

async def producer(queue: asyncio.Queue, n_items: int, name: str):
    """Produce n_items and put them on the queue."""
    for i in range(n_items):
        item = f'{name}-item-{i}'
        await asyncio.sleep(random.uniform(0.05, 0.15))   # simulate work
        await queue.put(item)
        print(f'  Produced: {item}  (queue size: {queue.qsize()})')
    await queue.put(None)   # sentinel: signal that this producer is done


async def consumer(queue: asyncio.Queue, consumer_id: int, n_producers: int):
    """Consume items from the queue until all producers are done."""
    done_count = 0
    while done_count < n_producers:
        item = await queue.get()
        if item is None:
            done_count += 1
            queue.task_done()
            continue
        await asyncio.sleep(random.uniform(0.01, 0.08))   # simulate processing
        print(f'  Consumer-{consumer_id} processed: {item}')
        queue.task_done()


async def pipeline():
    queue = asyncio.Queue(maxsize=5)   # bounded queue — back-pressure
    n_producers = 2

    # Two producers, two consumers run concurrently
    await asyncio.gather(
        producer(queue, 4, 'P1'),
        producer(queue, 3, 'P2'),
        consumer(queue, 1, n_producers),
        consumer(queue, 2, n_producers),
    )
    print('Pipeline complete.')


start = time.perf_counter()
asyncio.run(pipeline())
print(f'Elapsed: {time.perf_counter() - start:.2f}s')

## 7. `ThreadPoolExecutor` — CPU-Bound and Blocking I/O

asyncio is not suitable for **CPU-bound** tasks because a long computation blocks the event loop.  
Use `loop.run_in_executor()` to offload such work to a thread pool.

In [ ]:
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor

def cpu_bound_task(n: int) -> int:
    """Simulate a CPU-intensive computation (no asyncio inside)."""
    return sum(i * i for i in range(n))


async def async_cpu(n: int, executor: ThreadPoolExecutor) -> int:
    """Run the CPU task in a thread so the event loop stays free."""
    loop = asyncio.get_running_loop()
    return await loop.run_in_executor(executor, cpu_bound_task, n)


async def main():
    workloads = [2_000_000, 3_000_000, 1_000_000, 4_000_000]

    with ThreadPoolExecutor(max_workers=4) as pool:
        start = time.perf_counter()
        results = await asyncio.gather(*[async_cpu(n, pool) for n in workloads])
        elapsed = time.perf_counter() - start

    for n, r in zip(workloads, results):
        print(f'  sum_of_squares({n:,}) = {r:,}')
    print(f'Concurrent (thread pool): {elapsed:.2f}s')

asyncio.run(main())

## 8. When to Use What — Decision Guide

A summary of when to reach for asyncio, threads, or processes:

In [ ]:
# This cell prints a decision guide — no async needed here

guide = [
    ('I/O-bound, many connections (HTTP, DB, sockets)',
     'asyncio + aiohttp/asyncpg',
     'Single thread, thousands of concurrent connections, lowest overhead'),

    ('I/O-bound, small number of connections / legacy blocking code',
     'ThreadPoolExecutor',
     'Simple to add, reuses existing synchronous libraries'),

    ('CPU-bound, Python code (GIL-limited)',
     'ProcessPoolExecutor / multiprocessing',
     'Bypasses the GIL; separate processes with their own interpreter'),

    ('CPU-bound, numerical / scientific',
     'NumPy / Cython / Numba',
     'Release GIL inside compiled code; massive speedups'),

    ('Mixed I/O + light CPU processing',
     'asyncio + run_in_executor for CPU parts',
     'Keep event loop free; offload heavy computation to threads'),
]

print(f'{"Workload":<45} {"Tool":<35} {"Reason"}')
print('-' * 120)
for workload, tool, reason in guide:
    print(f'{workload:<45} {tool:<35} {reason}')

## Practice Exercises

**Exercise 1 — Async Rate-Limited Downloader**  
Write an async function `download_all(urls, max_concurrent=3)` that downloads a list of URLs using `aiohttp` but limits concurrent requests to `max_concurrent` using `asyncio.Semaphore`. Return a list of `(url, status_code, body_length)` tuples.

**Exercise 2 — Async Retry Decorator**  
Write an `async_retry(max_attempts=3, delay=0.5)` decorator for async functions. It should catch any exception, wait `delay` seconds (using `asyncio.sleep`), and retry up to `max_attempts` times before re-raising. Test it on an async function that fails randomly.

**Exercise 3 — Pipeline Benchmark**  
Simulate a data pipeline that: (1) reads 20 items from a source (each takes 0.05s), (2) transforms each item (0.02s), (3) writes results to a sink (0.03s). Implement it three ways: sequential, with `asyncio.gather`, and with a producer/consumer queue. Compare the total elapsed times.